# 06-03 샘플링으로 문장 생성하기

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요. 06-02에서 저장한 gpt.pt가 없으면 다시 학습해요.

In [ ]:
import json
import math
import os
import random
import time
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 0. 준비: 06-02의 데이터와 모델 코드

In [ ]:
URL = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/klue-nli-v1.1/klue-nli-v1.1_train.json"
if not os.path.exists("klue_nli_train.json"):
    urllib.request.urlretrieve(URL, "klue_nli_train.json")
with open("klue_nli_train.json", encoding="utf-8") as f:
    data = json.load(f)
sentences = sorted({s.strip() for d in data for s in (d["premise"], d["hypothesis"])})
random.seed(0)
random.shuffle(sentences)
text = "\n".join(sentences) + "\n"

chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = chars
V = len(chars)
data_ids = torch.tensor([stoi[ch] for ch in text])
n = int(len(data_ids) * 0.9)
train_ids, val_ids = data_ids[:n], data_ids[n:]

def encode(s):
    return [stoi[ch] for ch in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

def get_batch(ids, batch_size, block_size):
    starts = torch.randint(0, len(ids) - block_size - 1, (batch_size,))
    x = torch.stack([ids[s : s + block_size] for s in starts])  # (B, block_size)
    y = torch.stack([ids[s + 1 : s + block_size + 1] for s in starts])  # (B, block_size)
    return x.to(device), y.to(device)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_len):
        super().__init__()
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)  # q, k, v를 한 번에 계산해요
        self.out_proj = nn.Linear(d_model, d_model)
        causal = torch.tril(torch.ones(max_len, max_len, dtype=torch.bool))  # (max_len, max_len)
        self.register_buffer("causal", causal)  # 학습하지 않는 텐서. model.to(device)로 함께 옮겨져요

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)  # 각각 (B, T, C)
        q, k, v = (t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in (q, k, v))  # (B, H, T, d_head)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)  # (B, H, T, T)
        scores = scores.masked_fill(~self.causal[:T, :T], float("-inf"))  # 미래는 가려요
        self.weights = F.softmax(scores, dim=-1)  # (B, H, T, T)
        out = (self.weights @ v).transpose(1, 2).reshape(B, T, C)  # (B, T, C)
        return self.out_proj(out)


class Block(nn.Module):
    def __init__(self, d_model, n_heads, max_len, dropout):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len)
        self.ff = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model), nn.Dropout(dropout))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        return x + self.ff(self.norm2(x))  # (B, T, d_model)


class GPT(nn.Module):
    def __init__(self, vocab_size, d_model=192, n_heads=6, n_layers=4, max_len=64, dropout=0.1):
        super().__init__()
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, max_len, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)  # 다음 토큰 점수

    def forward(self, ids):
        # ids: (B, T)
        T = ids.shape[1]
        x = self.tok_emb(ids) + self.pos_emb(torch.arange(T, device=ids.device))  # (B, T, d_model)
        for block in self.blocks:
            x = block(x)  # (B, T, d_model)
        return self.lm_head(self.norm(x))  # (B, T, vocab_size)


@torch.no_grad()
def estimate_loss(model, ids, iters=50):
    model.eval()
    losses = []
    for _ in range(iters):
        x, y = get_batch(ids, 64, model.max_len)
        losses.append(F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1)).item())
    model.train()
    return sum(losses) / len(losses)


torch.manual_seed(0)
model = GPT(V).to(device)
if os.path.exists("gpt.pt"):
    model.load_state_dict(torch.load("gpt.pt", map_location=device))
    print("gpt.pt를 불러왔어요")
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    start = time.time()
    for step in range(3001):
        x, y = get_batch(train_ids, 64, model.max_len)
        loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 1000 == 0:
            print(f"step {step}: train loss={loss.item():.3f} ({time.time() - start:.0f}s)")
    torch.save(model.state_dict(), "gpt.pt")
print("val loss:", round(estimate_loss(model, val_ids), 3))
_ = model.eval()  # 생성할 때는 드롭아웃을 꺼요

## 1. 다음 토큰 확률 살펴보기

In [ ]:
@torch.no_grad()
def next_token_logits(prompt):
    ids = torch.tensor([encode(prompt)], device=device)[:, -model.max_len :]  # (1, T)
    return model(ids)[0, -1].cpu()  # (V,)

logits = next_token_logits("오늘 날씨가 ")
probs = F.softmax(logits, dim=-1)
top = probs.topk(10)
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"{itos[i]!r}: {p:.3f}")

## 2. 그리디: 항상 가장 높은 것

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=80, temperature=1.0, top_k=None, greedy=False, seed=0):
    g = torch.Generator().manual_seed(seed)
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        logits = next_token_logits(decode(ids))  # (V,)
        if greedy:
            nxt = logits.argmax().item()
        else:
            logits = logits / temperature  # 온도로 나누기
            if top_k is not None:
                kth = logits.topk(top_k).values[-1]
                logits = logits.masked_fill(logits < kth, float("-inf"))  # 상위 k개만 남기기
            nxt = torch.multinomial(F.softmax(logits, dim=-1), 1, generator=g).item()
        ids.append(nxt)
        if itos[nxt] == "\n":
            break
    return decode(ids).strip()

for prompt in ["오늘", "정부는"]:
    print(generate(prompt, greedy=True))

## 3. 온도(temperature)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
top_ids = probs.topk(15).indices
for ax, t in zip(axes, [0.5, 1.0, 2.0]):
    p = F.softmax(logits / t, dim=-1)[top_ids]
    ax.bar(range(15), p, color="tab:orange")
    ax.set_ylim(0, 1)
    ax.set_title(f"temperature = {t}")
    ax.set_xlabel("top-15 token rank")
plt.show()

In [ ]:
for t in [0.5, 1.0, 1.5]:
    print(f"[온도 {t}]")
    for seed in range(3):
        print(" ", generate("오늘", temperature=t, seed=seed))

## 4. top-k 샘플링

In [ ]:
for k in [1, 5, 50]:
    print(f"[top-k {k}]")
    for seed in range(3):
        print(" ", generate("정부는", top_k=k, seed=seed))

## 5. 직접 해 보기

In [ ]:
print(generate("이 영화는", temperature=0.8, top_k=20, seed=0))